# Phase 10.5D: Forecasting Architecture Experiments

This notebook analyzes the results of the 5 forecasting architecture experiments recorded in the **Experiment Registry** (`data/models/experiments/`).

### Strict Protocol Constraints:
- **Zero Test Set Leakage**: Evaluated exclusively using **3-Fold Expanding Chronological Cross-Validation** on the training partition (`X_train_v2` / `y_train_v2`).
- **Feature Set**: Weather-Enriched v2 features (114 predictors).
- **Architectures Evaluated**:
  1. **EXP-015**: Multi-Pollutant Regression ($PM_{2.5}, PM_{10}, O_3, NO_2, SO_2, CO$) $\to$ Deterministic US EPA AQI Piecewise Conversion
  2. **EXP-016**: Grouped Horizon Ridge Regressor ($h1-6, h7-24, h25-72$)
  3. **EXP-017**: **Hybrid AQI Specialist** (LightGBM $h1-6$ + Ridge $h7-72$)
  4. **EXP-018**: Hybrid Multi-Pollutant Specialist (LightGBM $h1-6$ + Ridge $h7-72$ on 6 pollutants $\to$ EPA AQI)
  5. **EXP-019**: **Persistence-Aware Hybrid Ensemble** (LightGBM $h1-6$ + Ridge $h7-37$ + Blended Ridge/Persistence $h38-72$)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.training_pipeline.experiment_registry import ExperimentRegistry

sns.set_theme(style="whitegrid")
registry = ExperimentRegistry()
df_leaderboard = registry.get_leaderboard()
df_leaderboard

## 1. Overall Validation Leaderboard Across All Experiments (EXP-001 to EXP-019)

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(
    data=df_leaderboard,
    x="Val RMSE (Mean)",
    y="Experiment ID",
    hue="Model",
    dodge=False,
    palette="tab20"
)
plt.title("Cross-Validation RMSE Leaderboard Across All Model & Architecture Experiments", fontsize=14, fontweight="bold")
plt.xlabel("Mean Validation RMSE (Lower is Better)")
plt.ylabel("Experiment ID")
plt.xlim(80, 100)
plt.tight_layout()
plt.show()

## 2. Multi-Horizon Breakdown: Short (h+1), Medium (h+24), and Long (h+72)

In [ ]:
arch_exps = ["EXP-005", "EXP-013", "EXP-015", "EXP-016", "EXP-017", "EXP-019"]
df_arch = df_leaderboard[df_leaderboard["Experiment ID"].isin(arch_exps)].copy()

horizon_rows = []
for _, row in df_arch.iterrows():
    horizon_rows.append({"Architecture": f"{row['Experiment ID']}: {row['Model']}", "Horizon": "h+1 (1h)", "RMSE": row["h+1 RMSE"]})
    horizon_rows.append({"Architecture": f"{row['Experiment ID']}: {row['Model']}", "Horizon": "h+24 (24h)", "RMSE": row["h+24 RMSE"]})
    horizon_rows.append({"Architecture": f"{row['Experiment ID']}: {row['Model']}", "Horizon": "h+72 (72h)", "RMSE": row["h+72 RMSE"]})

df_horizons = pd.DataFrame(horizon_rows)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_horizons, x="Horizon", y="RMSE", hue="Architecture", palette="Dark2")
plt.title("Per-Horizon Error Profile Across Forecasting Architectures", fontsize=14, fontweight="bold")
plt.ylabel("Validation RMSE")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Key Scientific Conclusions from Phase 10.5D

1. **EXP-017 (Hybrid Specialist) is the Clear Overall Winner**:
   - Combining **LightGBM Direct for $h1-6$** with **Ridge Regression for $h7-72$** achieves the best overall validation performance across the entire project:
     - **Val RMSE = 85.46** (outperforming all standalone models).
     - **$h+1$ RMSE = 46.18** (matches LightGBM's state-of-the-art short-horizon precision, beating Ridge's $52.66$ by **12.3%**).
     - **$h+72$ RMSE = 92.39** (maintains Ridge's strong regularization against long-horizon tree variance).
2. **EXP-019 (Persistence-Aware Hybrid)**:
   - Delivers the lowest mean MAE (**62.95**), confirming that long-horizon linear blending with persistence stabilizes absolute deviations.
3. **Multi-Pollutant $\to$ EPA AQI Dynamics (EXP-015 & EXP-018)**:
   - Forecasting individual pollutant concentrations before taking the $\max(\cdot)$ sub-index operator suffers from **error compounding in the max function**: when forecasting 6 pollutants simultaneously, independent positive estimation noise in any single pollutant elevates the maximum, resulting in a positive bias ($RMSE = 92.19$).
4. **Locked Candidates for Phase 10.5E Test Benchmark**:
   - Primary Candidate: **EXP-017 (Hybrid AQI Specialist)**
   - Secondary Candidate: **EXP-005 (Ridge Regression $\alpha=1.0$ with Weather Telemetry)**
   - Ensemble Candidate: **EXP-019 (Persistence-Aware Hybrid)**